# Notebook 8: Agent Guardrails - SHACL as a Steering Hook

## Why Agent Guardrails?

Throughout this workshop, we've built a semantic reasoning pipeline: ontology → data → SHACL validation → SPARQL queries → LLM explanations. Each step has been **human-driven** - you ran the cells, reviewed the output, decided what to do next.

In production, these capabilities become **tools that AI agents call autonomously**. An agent might decide to send a denial notice, trigger a payment, or escalate a claim - all without human review. This raises a critical question:

> **How do you prevent an agent from taking an irreversible action on unvalidated data?**

The answer: **steering hooks** - interception points in the agent's execution loop where policy checks run before any tool executes.

## What is Strands?

[Strands Agents](https://github.com/strands-agents/sdk-python) is an open-source Python SDK for building AI agents. It provides:

- **Tools** - Python functions the agent can call (search, send email, query a database)
- **Models** - LLM backends (Amazon Bedrock, OpenAI, etc.) that drive the agent's reasoning
- **Plugins & Hooks** - interception points in the agent lifecycle where you can inject custom logic

The key concept for this notebook is the **`BeforeToolCallEvent` hook**. Every time the agent decides to call a tool, Strands fires this event *before* the tool executes. A plugin can inspect the call, validate it, and either allow it to proceed or cancel it.

## The Pattern: SHACL as a Steering Hook

```
Agent decides to call send_denial_notice(claim_id="C003")
         │
         ▼
┌─────────────────────────────┐
│   BeforeToolCallEvent       │
│                             │
│   ShaclGuardPlugin:         │
│   1. Load claim from graph  │
│   2. Run SHACL eligibility  │
│   3. Violations found?      │
│      YES → CANCEL the call  │
│      NO  → PROCEED          │
└─────────────┬───────────────┘
              │
         (if allowed)
              ▼
┌─────────────────────────────┐
│   send_denial_notice()      │
│   (executes normally)       │
└─────────────────────────────┘
```

The **tool is clean** - it just sends a notice, with no validation logic. The **plugin enforces policy** - it runs SHACL shapes as a pre-execution gate. The **shapes are declarative** - swap them to change the policy without touching the tool or the agent.

This separation means:
- Business analysts own the SHACL shapes (policy)
- Developers own the tools (capabilities)
- The agent framework wires them together (orchestration)


---
### Setup


In [ ]:
%graph_notebook_config --store-to NOTEBOOK_CONFIG --silent

In [ ]:
from workshop import *
import json
configure(neptune_config=json.loads(NOTEBOOK_CONFIG))

g = load_ontology()
data_graph, _ = load_claim_data()
g += data_graph
BEDROCK_REGION = "us-east-1"
SHAPES_PATH = str(NOTEBOOK_DIR / "eligibility_shapes.ttl")

### Imports and SHACL Gate Function

The `shacl_gate` function is the core of the guardrail - it takes a claim URI, runs SHACL eligibility validation against the data graph, and returns whether the claim passes or fails. This is the same validation from Notebook 02, now packaged as a reusable gate:


In [ ]:
# ── 11.1: Imports & SHACL gate function ───────────────────────────────

try:
    from strands import Agent, tool
    from strands.models import BedrockModel
    from strands.plugins import Plugin, hook
    from strands.hooks import BeforeToolCallEvent
    STRANDS_AVAILABLE = True
except ImportError:
    STRANDS_AVAILABLE = False
    print("strands SDK not installed — run: pip install strands-agents strands-agents-bedrock")
    print("This module will show the code but skip execution.")

from pyshacl import validate as shacl_validate
from rdflib import Graph as RdfGraph, Namespace, URIRef

INS = Namespace("http://example.org/insurance#")
SHAPES_PATH = str(NOTEBOOK_DIR / "eligibility_shapes.ttl")


def shacl_gate(claim_id: str, data_graph, shapes_path: str) -> dict:
    """Run SHACL eligibility validation for a single claim.
    Returns {allowed: bool, violations: list[str]}."""
    shapes = RdfGraph()
    shapes.parse(shapes_path)

    conforms, results_graph, results_text = shacl_validate(
        data_graph, shacl_graph=shapes,
        advanced=True, allow_infos=True,
    )

    # Map claim ID to URI: C003 → claim003
    claim_num = claim_id.replace("C", "")
    claim_uri = f"{INS}claim{claim_num.zfill(3)}"

    violations = []
    for line in results_text.strip().split("\n"):
        if claim_uri in line or claim_id in line:
            violations.append(line.strip())

    if not violations:
        for s, p, o in results_graph:
            if str(s) == claim_uri or str(o) == claim_uri:
                violations.append(f"{s} {p} {o}")

    return {"allowed": len(violations) == 0, "violations": violations}


print("SHACL gate function ready.")
print("Strands SDK:", "available" if STRANDS_AVAILABLE else "not installed")

### Plugin, Tool, and Agent

The `ShaclGuardPlugin` intercepts `BeforeToolCallEvent` - the moment before any tool executes. It extracts the claim ID from the tool arguments, runs the SHACL gate, and either allows the call to proceed or cancels it with an error message. The agent never sees the guardrail - it just gets told the action was blocked:


In [ ]:
# ── 11.2: Plugin, Tool, and Agent ─────────────────────────────────────

if STRANDS_AVAILABLE:

    # ── Plugin: intercepts BeforeToolCallEvent ────────────────────
    class ShaclGuardPlugin(Plugin):
        """Blocks tool calls that fail SHACL eligibility validation."""
        name = "shacl-guard"

        def __init__(self, data_graph, shapes_path: str):
            super().__init__()
            self.data_graph = data_graph
            self.shapes_path = shapes_path

        @hook
        def before_tool(self, event: BeforeToolCallEvent) -> None:
            """Run SHACL validation before send_denial_notice executes."""
            tool_name = event.tool_use.get("name", "")
            if tool_name != "send_denial_notice":
                return  # only gate this specific tool

            claim_id = event.tool_use.get("input", {}).get("claim_id", "")
            print(f"  [ShaclGuardPlugin] Validating {claim_id}...")

            result = shacl_gate(claim_id, self.data_graph, self.shapes_path)

            if result["allowed"]:
                reason = (
                    f"BLOCKED: {claim_id} has no eligibility violations — "
                    f"no justified reason to send a denial notice. Review claim status."
                )
                print(f"  [ShaclGuardPlugin] {reason}")
                event.cancel_tool = reason
            else:
                print(f"  [ShaclGuardPlugin] {claim_id} has {len(result['violations'])} violation(s) — denial justified, proceeding.")


    # ── Tool: clean, no validation logic ────────────────────────────
    @tool
    def send_denial_notice(claim_id: str) -> str:
        """Send a denial notice to the patient for a specific claim.

        Args:
            claim_id: The claim identifier (e.g. 'C003', 'C001')
        """
        # In production this would call an email/notification service
        return f"Denial notice sent for {claim_id}. Patient notified."


    # ── Agent: wired with plugin ──────────────────────────────────
    model = BedrockModel(
        model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
        region_name=BEDROCK_REGION,
    )

    shacl_plugin = ShaclGuardPlugin(data_graph=g, shapes_path=SHAPES_PATH)

    claims_agent = Agent(
        model=model,
        tools=[send_denial_notice],
        plugins=[shacl_plugin],
        system_prompt=(
            "You are a claims processing agent. When asked to send a denial notice, "
            "call the send_denial_notice tool with the claim ID. Report the result "
            "to the user exactly as returned — do not fabricate outcomes."
        ),
    )

    print("Agent ready with ShaclGuardPlugin on BeforeToolCallEvent.")
else:
    print("Skipping agent creation — strands SDK not installed.")

### Test 1: Claim C003 (Coverage Gap - Justified Denial)

Claim C003 has a SHACL eligibility violation - the incident occurred during a coverage gap. The SHACL gate finds violations, which means there IS a valid reason to deny this claim. The plugin **allows** the tool call to proceed - the denial notice is justified.

Expected outcome: the plugin allows the tool call and the denial notice is sent.


In [ ]:
# ── 11.3: ALLOWED path — C003 (coverage gap = justified denial) ─────────────────────────
# Claim C003 has a coverage gap violation — the denial IS justified.
# The plugin finds violations, so sending the denial notice is ALLOWED.

if STRANDS_AVAILABLE:
    print("═" * 60)
    print("Test 1: C003 — coverage gap (expect ALLOWED)")
    print("═" * 60)
    response = claims_agent("Send a denial notice to the patient for claim C003")
    print(f"\nAgent response:\n{response}")
else:
    print("Strands not installed — running SHACL gate directly:")
    result = shacl_gate("C003", g, SHAPES_PATH)
    print(f"  allowed={result['allowed']}, violations={len(result['violations'])}")
    print(f"  → Plugin would PROCEED (denial is justified)")

### Test 2: Claim C001 (Clean Claim - No Justified Denial)

Claim C001 has no eligibility violations - it went through the normal lifecycle and was paid. The SHACL gate finds no violations, which means there is NO valid reason to deny this claim. The plugin **blocks** the tool call - you can't send a denial notice without a justified reason.

Expected outcome: the plugin blocks the tool call and the agent reports that the action was prevented.


In [ ]:
# ── 11.4: BLOCKED path — C001 (no violations = no justified denial) ──────────────────────────
# Claim C001 has no eligibility violations — there's no justified reason to deny.
# The plugin BLOCKS the notice because there's no valid denial reason.

if STRANDS_AVAILABLE:
    print("═" * 60)
    print("Test 2: C001 — clean claim (expect BLOCKED)")
    print("═" * 60)
    response = claims_agent("Send a denial notice to the patient for claim C001")
    print(f"\nAgent response:\n{response}")
else:
    print("Strands not installed — running SHACL gate directly:")
    result = shacl_gate("C001", g, SHAPES_PATH)
    print(f"  allowed={result['allowed']}, violations={len(result['violations'])}")
    print(f"  → Plugin would CANCEL (no justified denial)")

### What Just Happened

| Test | Claim | SHACL Result | Plugin Action | Tool Executed? |
|------|-------|--------------|---------------|----------------|
| 1 | C003 | Coverage gap violation | **CANCEL** — injected error result | No |
| 2 | C001 | No violations | **PROCEED** | Yes — notice sent |

The key design: the **tool is clean** — it just sends a notice. The **plugin enforces policy** — it intercepts `BeforeToolCallEvent`, runs SHACL, and cancels the call if violations exist. Neither knows about the other.

```python
# The separation:
send_denial_notice()    # pure action — no validation logic
ShaclGuardPlugin        # pure policy — no business logic
eligibility_shapes.ttl  # pure rules  — declarative SHACL
```

This pattern generalizes to any irreversible agent action (emails, payments, system updates). Swap the SHACL shapes to change the policy without touching the tool or the agent.

---
## Summary

### What You Built

**Pipeline 1 - Extract (Module 10):** Unstructured text → structured RDF via ontology-guided LLM

```
  Unstructured Document          Ontology (OWL)
         │                            │
         ▼                            ▼
       ┌────────────────────────────────┐
       │          Bedrock LLM           │
       │       (text → RDF extract)     │
       └──────────────┬─────────────────┘
                      │
               Extracted Triples
                      │
                      ▼
       ┌────────────────────────────────┐
       │         Graph Backend          │
       │  Neptune / Ontop VKG / rdflib  │
       └────────────────────────────────┘
```

**Pipeline 2 - Query & Explain (Modules 7–9):** Natural language → SPARQL → SHACL → denial explanation

```
  Natural Language Question
         │
         ▼
  ┌──────────────┐      ┌───────────────┐      ┌─────────────────────────────┐
  │  Bedrock LLM │─────▶│ SPARQL Query  │─────▶│        Graph Backend        │
  │ (text→SPARQL)│      └───────────────┘      │ Neptune / Ontop VKG / rdflib│
  └──────────────┘                             └──────────────┬──────────────┘
                                                              │
                                                         Claim Data
                                                              │
                                                              ▼
                                                      ┌────────────────┐
                                                      │  SHACL Shapes  │
                                                      │   (pyshacl)    │
                                                      └───────┬────────┘
                                                              │
                                                          Violations
                                                              │
                                                              ▼
                                                      ┌────────────────┐
                                                      │  Bedrock LLM   │
                                                      │(explain denial)│
                                                      └───────┬────────┘
                                                              │
                                                              ▼
                                                       Customer-Facing
                                                      Denial Explanation
```

### Key Takeaways

1. **Ontology as schema** - OWL defines the vocabulary; SHACL enforces constraints
2. **SCD2 in RDF** - Temporal coverage history enables gap detection without SQL
3. **Business rules as shapes** - Eligibility logic is declarative, testable, and auditable
4. **No hardcoded denial codes** - SHACL violations ARE the denial reasons
5. **LLM as translator** - Bedrock converts machine-readable violations into human language
6. **Triple backend** - Same SPARQL queries work against Neptune, Ontop VKG, or rdflib
7. **Virtual Knowledge Graph** - Ontop exposes relational data as RDF with zero ETL
8. **Ontology-guided extraction** - LLM extracts structured RDF from unstructured text, constrained by the ontology to prevent hallucinated predicates
9. **Agent guardrails** - SHACL validation as a pre-tool hook prevents agents from acting on unvalidated data